# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema accessible at the URL below. All entities (such as record sets, fields, and columns) are referenced by their `@id` as per FAIR best practices.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL (JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
print(f"Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}\n")
print(f"Published: {dataset.metadata.datePublished}\n")

## 2. Data Overview
List all record sets, their `@id`, fields, and columns descriptions provided by the Croissant schema.

In [ ]:
print("Available record sets:")
record_sets = list(dataset.record_sets())
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}\n")

    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name} | Description: {field.description}")

    print("  Columns:")
    for column in record_set.columns:
        print(f"    - Column @id: {column.id} | Name: {column.name} | DataType: {column.data_type}")
    print()

## 3. Data Extraction
Load data from each record set into pandas DataFrames for further exploration. All references are by `@id`.

In [ ]:
# Retrieve all record set IDs
record_set_ids = [record_set.id for record_set in dataset.record_sets()]
print(f"Record set IDs: {record_set_ids}")

# Extract records from each record set
dataframes = {}
for record_set_id in record_set_ids:
    # Note: The mlcroissant dataset.records API expects the `@id` of a record set.
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
    print(f"Fields: {df.columns.tolist()}\n")
    # Display first 2 records as a preview
    print(df.head(2))


## 4. Exploratory Data Analysis (EDA)
Let's choose the main table (often the largest record set) for EDA. We select a numeric field (such as peptide counts, molecular weight, or protein abundance), filter records by value, normalize it, and group by a categorical field, all by their `@id`.

In [ ]:
# --- Customize these as found in Section 2 outputs ---
# Select a main record set by @id (update as needed based on data overview; commonly the 1st)
main_record_set_id = record_set_ids[0]

# Select a numeric field by @id or column name (update as appropriate)
main_df = dataframes[main_record_set_id]

# Display columns
print(main_df.columns)

# Example: suppose 'cr:peptideCount' is a protein peptide count field (use actual field @id or column name)
numeric_field_id = None
for col in main_df.columns:
    if 'peptide' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # fallback to first numeric column, if available
    for col in main_df.select_dtypes(include='number').columns:
        numeric_field_id = col
        break
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Apply thresholding (e.g., peptide counts > threshold)
threshold = 10
if numeric_field_id is None:
    print('No suitable numeric field found for EDA.')
    filtered_df = main_df
else:
    # Convert to numeric (in case it's read as object)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records in '{main_record_set_id}' where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (e.g., 'cr:description', or category field)
    group_field_id = None
    for col in main_df.columns:
        if ('description' in col.lower() or 'category' in col.lower()) and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between numeric fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded schema and data from the FAIR^2 dataset using `mlcroissant`.
- Explored its available record sets, fields, and columns by their unique `@id`.
- Loaded records into pandas DataFrames for processing and analysis.
- Demonstrated filtering, normalization, grouping, and visualization of a key numeric field.

Refer to the Croissant schema original documentation for more information on entity definitions, and further tailor these analyses for your specific research questions.